# E-Commerce Customer Insights & Churn Analytics

Complete data science workflow: Descriptive → Diagnostic → Predictive → Prescriptive

2,000 customer records across 17 columns. Sales performance, customer behavior, churn identification, and a predictive model for at-risk customers.

## Setup & Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report, roc_curve
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 120

## Data Loading & Inspection

Load the dataset and do a basic inspection.

In [ ]:
df = pd.read_csv('data/E Commerce Customer Insights and Churn Dataset.csv')
df.head()
df.shape
df.info()
df.describe()

## Data Quality Audit

Check for missing values, duplicates, unique counts, and date consistency.

In [ ]:
# Basic checks
print("Missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
print("Unique customers:", df['customer_id'].nunique())
print("Unique orders:", df['order_id'].nunique())
print("Orders per customer:", df.groupby('customer_id')['order_id'].count().unique())

# Parse dates
df['signup_date'] = pd.to_datetime(df['signup_date'])
df['order_date'] = pd.to_datetime(df['order_date'])
df['last_purchase_date'] = pd.to_datetime(df['last_purchase_date'])

print(f"Signup range: {df['signup_date'].min().date()} to {df['signup_date'].max().date()}")
print(f"Order range: {df['order_date'].min().date()} to {df['order_date'].max().date()}")
print(f"Last purchase range: {df['last_purchase_date'].min().date()} to {df['last_purchase_date'].max().date()}")
print(f"All signups before orders: {(df['signup_date'] <= df['order_date']).all()}")

## Feature Engineering

Create derived features and define the churn target.

In [ ]:
# Revenue per order
df['revenue'] = df['unit_price'] * df['quantity']

# Customer tenure
df['tenure_days'] = (df['order_date'] - df['signup_date']).dt.days

# Recency relative to latest purchase date
analysis_date = df['last_purchase_date'].max()
df['recency_days'] = (analysis_date - df['last_purchase_date']).dt.days

# Churn target: cancelled = 1, active/paused = 0
df['churn'] = (df['subscription_status'] == 'cancelled').astype(int)

print(f"Churn rate: {df['churn'].mean()*100:.1f}%")
df[['revenue', 'tenure_days', 'recency_days', 'churn']].head()

## Sales Performance

Key revenue metrics and breakdowns by country, category, and subscription status.

In [ ]:
total_revenue = df['revenue'].sum()
aov = df['revenue'].mean()

print(f"Total Revenue: ${total_revenue:,.2f}")
print(f"Average Order Value: ${aov:,.2f}")

# Revenue by country
country_rev = df.groupby('country')['revenue'].agg(['sum', 'mean', 'count'])
country_rev.columns = ['Total Revenue', 'Avg Revenue', 'Orders']
country_rev.sort_values('Total Revenue', ascending=False)

# Revenue by category
cat_rev = df.groupby('category')['revenue'].agg(['sum', 'mean', 'count'])
cat_rev.columns = ['Total Revenue', 'Avg Revenue', 'Orders']
cat_rev.sort_values('Total Revenue', ascending=False)

# Monthly revenue
df['order_month'] = df['order_date'].dt.to_period('M')
monthly_revenue = df.groupby('order_month')['revenue'].sum()
monthly_revenue.head(12)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0,0].hist(df['revenue'], bins=30, color='#3498db', edgecolor='black', alpha=0.7)
axes[0,0].set_title('Revenue Distribution')
axes[0,0].set_xlabel('Revenue ($)')

df.groupby('country')['revenue'].sum().sort_values(ascending=True).plot(kind='barh', ax=axes[0,1], color='#2ecc71')
axes[0,1].set_title('Revenue by Country')

df.groupby('category')['revenue'].sum().sort_values(ascending=True).plot(kind='barh', ax=axes[1,0], color='#e74c3c')
axes[1,0].set_title('Revenue by Category')

monthly_revenue.plot(ax=axes[1,1], color='#9b59b6', marker='o')
axes[1,1].set_title('Monthly Revenue Trend')
axes[1,1].set_xlabel('Month')
plt.xticks(rotation=45)

plt.tight_layout()
plt.savefig('outputs/figures/sales_performance.png', dpi=150, bbox_inches='tight')
plt.show()

## Customer Analysis

Demographics, subscription status, and cancellation patterns.

In [ ]:
print("Age describe:")
df['age'].describe()

print(f"\nAge groups:")
df['age_group'] = pd.cut(df['age'], bins=[17, 25, 35, 45, 55, 70], labels=['18-25', '26-35', '36-45', '46-55', '56-70'])
df['age_group'].value_counts().sort_index()

print(f"\nGender:")
df['gender'].value_counts()

print(f"\nCountry:")
df['country'].value_counts()

print(f"\nSubscription status:")
df['subscription_status'].value_counts()

print(f"\nCancellations:")
df['cancellations_count'].value_counts().sort_index()

print(f"\nPurchase frequency range: {df['purchase_frequency'].min()} to {df['purchase_frequency'].max()}")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

df['age'].plot(kind='hist', bins=20, ax=axes[0,0], color='#3498db', edgecolor='black', alpha=0.7)
axes[0,0].set_title('Age Distribution')

df['gender'].value_counts().plot(kind='pie', ax=axes[0,1], autopct='%1.1f%%', colors=['#e74c3c','#3498db','#2ecc71'])
axes[0,1].set_title('Gender')
axes[0,1].set_ylabel('')

df['country'].value_counts().plot(kind='bar', ax=axes[0,2], color='#9b59b6')
axes[0,2].set_title('Country')
plt.xticks(rotation=45)

df['subscription_status'].value_counts().plot(kind='bar', ax=axes[1,0], color=['#2ecc71','#e74c3c','#f39c12'])
axes[1,0].set_title('Subscription Status')
plt.xticks(rotation=0)

df['purchase_frequency'].plot(kind='hist', bins=30, ax=axes[1,1], color='#e67e22', edgecolor='black', alpha=0.7)
axes[1,1].set_title('Purchase Frequency')

df['cancellations_count'].value_counts().sort_index().plot(kind='bar', ax=axes[1,2], color='#1abc9c')
axes[1,2].set_title('Cancellations')
plt.xticks(rotation=0)

plt.tight_layout()
plt.savefig('outputs/figures/customer_demographics.png', dpi=150, bbox_inches='tight')
plt.show()

## Diagnostic Analysis

Correlation analysis and churn rate breakdowns.

In [ ]:
numeric_cols = ['age', 'unit_price', 'quantity', 'revenue', 'tenure_days', 'recency_days', 'purchase_frequency', 'cancellations_count', 'churn']
corr_matrix = df[numeric_cols].corr()
corr_matrix['churn'].sort_values(ascending=False)

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r', center=0, square=True, linewidths=0.5)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.savefig('outputs/figures/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Churn rates by category, country, gender, cancellations
for cat in df['category'].unique():
    print(f"{cat}: {df[df['category']==cat]['churn'].mean()*100:.1f}% churn")

print()
for ctry in sorted(df['country'].unique()):
    print(f"{ctry}: {df[df['country']==ctry]['churn'].mean()*100:.1f}% churn")

print()
for g in df['gender'].unique():
    print(f"{g}: {df[df['gender']==g]['churn'].mean()*100:.1f}% churn")

print()
for c in sorted(df['cancellations_count'].unique()):
    print(f"{c} cancellations: {df[df['cancellations_count']==c]['churn'].mean()*100:.1f}% churn")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

df.groupby('category')['churn'].mean().sort_values().plot(kind='bar', ax=axes[0], color='#e74c3c')
axes[0].set_title('Churn by Category')
plt.xticks(rotation=45)

df.groupby('country')['churn'].mean().sort_values().plot(kind='bar', ax=axes[1], color='#3498db')
axes[1].set_title('Churn by Country')
plt.xticks(rotation=45)

df.groupby('cancellations_count')['churn'].mean().plot(kind='bar', ax=axes[2], color='#2ecc71')
axes[2].set_title('Churn by Cancellations')
plt.xticks(rotation=0)

plt.tight_layout()
plt.savefig('outputs/figures/churn_diagnostics.png', dpi=150, bbox_inches='tight')
plt.show()

## RFM Segmentation

Recency, Frequency, Monetary scoring using `last_purchase_date` for recency, `purchase_frequency` for frequency, and `revenue` for monetary.

In [ ]:
df['R_score'] = pd.qcut(df['recency_days'], q=4, labels=[4, 3, 2, 1]).astype(int)
df['F_score'] = pd.qcut(df['purchase_frequency'], q=4, labels=[1, 2, 3, 4]).astype(int)
df['M_score'] = pd.qcut(df['revenue'], q=4, labels=[1, 2, 3, 4]).astype(int)
df['RFM_score'] = df['R_score'] + df['F_score'] + df['M_score']

def segment_customer(rfm):
    if rfm >= 10: return 'Champions'
    elif rfm >= 8: return 'Loyal Customers'
    elif rfm >= 6: return 'Potential Loyalists'
    elif rfm >= 4: return 'At Risk'
    else: return 'Hibernating'

df['segment'] = df['RFM_score'].apply(segment_customer)

df['segment'].value_counts()

for seg in ['Champions', 'Loyal Customers', 'Potential Loyalists', 'At Risk', 'Hibernating']:
    subset = df[df['segment'] == seg]
    print(f"{seg}: {len(subset)} customers, avg revenue ${subset['revenue'].mean():,.2f}, avg churn {subset['churn'].mean()*100:.1f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df['segment'].value_counts().sort_values().plot(kind='barh', ax=axes[0], color='#3498db')
axes[0].set_title('Customer Segments (RFM)')

df.groupby('segment')['churn'].mean().sort_values().plot(kind='bar', ax=axes[1], color='#e74c3c')
axes[1].set_title('Churn Rate by Segment')
plt.xticks(rotation=45)

plt.tight_layout()
plt.savefig('outputs/figures/rfm_segments.png', dpi=150, bbox_inches='tight')
plt.show()

## Churn Analysis

Compare churned vs active customers across key metrics.

In [ ]:
churned = df[df['churn'] == 1]
active = df[df['churn'] == 0]

print(f"Churn rate: {df['churn'].mean()*100:.1f}%")
print(f"Churned: {len(churned)}, Active/Paused: {len(active)}")

# Compare means
metrics = ['revenue', 'tenure_days', 'recency_days', 'purchase_frequency', 'cancellations_count', 'age']
comp = pd.DataFrame({
    'Churned': [churned[m].mean() for m in metrics],
    'Active': [active[m].mean() for m in metrics]
}, index=['Revenue', 'Tenure', 'Recency', 'Freq', 'Cancels', 'Age'])
comp

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0,0].hist(churned['revenue'], bins=20, alpha=0.5, color='#e74c3c', label='Churned')
axes[0,0].hist(active['revenue'], bins=20, alpha=0.5, color='#2ecc71', label='Active')
axes[0,0].set_title('Revenue: Churned vs Active')
axes[0,0].legend()

axes[0,1].hist(churned['tenure_days'], bins=20, alpha=0.5, color='#e74c3c', label='Churned')
axes[0,1].hist(active['tenure_days'], bins=20, alpha=0.5, color='#2ecc71', label='Active')
axes[0,1].set_title('Tenure: Churned vs Active')
axes[0,1].legend()

axes[1,0].hist(churned['recency_days'], bins=20, alpha=0.5, color='#e74c3c', label='Churned')
axes[1,0].hist(active['recency_days'], bins=20, alpha=0.5, color='#2ecc71', label='Active')
axes[1,0].set_title('Recency: Churned vs Active')
axes[1,0].legend()

df.groupby('cancellations_count')['churn'].mean().plot(kind='bar', ax=axes[1,1], color='#9b59b6')
axes[1,1].set_title('Churn by Cancellations Count')
plt.xticks(rotation=0)

plt.tight_layout()
plt.savefig('outputs/figures/churn_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## Data Preprocessing for Modeling

One-hot encode categorical features and scale numerical features using StandardScaler fitted on training data only.

In [ ]:
feature_cols = ['age', 'tenure_days', 'recency_days', 'purchase_frequency', 'cancellations_count', 'unit_price', 'quantity', 'revenue', 'country', 'gender', 'preferred_category', 'category']
X = df[feature_cols].copy()
y = df['churn'].copy()

# One-hot encode categorical features
X_encoded = pd.get_dummies(X, columns=['country', 'gender', 'preferred_category', 'category'], drop_first=True)
print(f"Features: {X_encoded.shape[1]} columns after encoding")
print(f"Target distribution:\n{y.value_counts(normalize=True)}")

In [ ]:
# Train/test split with stratification
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42, stratify=y)

# Scale using training data only (prevent leakage)
scaler = StandardScaler()
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train: {X_train.shape[0]}, Test: {X_test.shape[0]}")
print(f"Train churn rate: {y_train.mean()*100:.1f}%")
print(f"Test churn rate: {y_test.mean()*100:.1f}%")

## Predictive Modeling

Train a Logistic Regression model to predict churn.

In [ ]:
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")
print(classification_report(y_test, y_pred, target_names=['Active/Paused', 'Churned']))

## Model Evaluation

Confusion matrix and ROC curve, plus feature importance.

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Active/Paused', 'Churned'],
            yticklabels=['Active/Paused', 'Churned'])
axes[0].set_title('Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

axes[1].plot(fpr, tpr, color='#3498db', linewidth=2, label=f'Logistic Regression (AUC = {roc_auc:.4f})')
axes[1].plot([0, 1], [0, 1], color='#999', linestyle='--', linewidth=1)
axes[1].fill_between(fpr, tpr, alpha=0.1, color='#3498db')
axes[1].set_title('ROC Curve')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend()

plt.tight_layout()
plt.savefig('outputs/figures/model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
feature_names = list(X_encoded.columns)
coefficients = pd.DataFrame({
    'feature': feature_names,
    'coefficient': model.coef_[0]
})
coefficients['abs_coef'] = coefficients['coefficient'].abs()
coefficients = coefficients.sort_values('abs_coef', ascending=False).drop('abs_coef', axis=1)

print("Top features by coefficient magnitude:")
coefficients.head(10)

## Risk Segmentation

Apply the model to all customers to generate churn probabilities and risk segments.

In [ ]:
df['churn_probability'] = model.predict_proba(X_encoded)[:, 1]

def risk_segment(prob):
    if prob >= 0.7: return 'High Risk'
    elif prob >= 0.4: return 'Medium Risk'
    else: return 'Low Risk'

df['risk_segment'] = df['churn_probability'].apply(risk_segment)

df['risk_segment'].value_counts()

for seg in ['High Risk', 'Medium Risk', 'Low Risk']:
    subset = df[df['risk_segment'] == seg]
    print(f"{seg}: {len(subset)} customers, churn rate {subset['churn'].mean()*100:.1f}%")

high_risk = df[df['risk_segment'] == 'High Risk']
print(f"\nHigh Risk revenue: ${high_risk['revenue'].sum():,.2f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

df['risk_segment'].value_counts().plot(kind='bar', ax=axes[0], color=['#2ecc71','#f39c12','#e74c3c'])
axes[0].set_title('Risk Distribution')
plt.xticks(rotation=0)

axes[1].hist(df[df['churn']==1]['churn_probability'], bins=20, alpha=0.5, color='#e74c3c', label='Churned')
axes[1].hist(df[df['churn']==0]['churn_probability'], bins=20, alpha=0.5, color='#2ecc71', label='Active')
axes[1].set_title('Churn Probability Distribution')
axes[1].legend()

top50 = df.nlargest(50, 'churn_probability')
colors = {'Low Risk': '#2ecc71', 'Medium Risk': '#f39c12', 'High Risk': '#e74c3c'}
for seg in top50['risk_segment'].unique():
    subset = top50[top50['risk_segment'] == seg]
    axes[2].scatter(subset['revenue'], subset['churn_probability'], c=colors[seg], label=seg, alpha=0.5, s=40)
axes[2].set_title('Revenue vs Churn Probability')
axes[2].set_xlabel('Revenue ($)')
axes[2].set_ylabel('Churn Probability')
axes[2].legend(fontsize='small')

plt.tight_layout()
plt.savefig('outputs/figures/risk_segmentation.png', dpi=150, bbox_inches='tight')
plt.show()

## Key Business Insights

In [ ]:
print("Revenue Concentration:")
top_20 = df.nlargest(int(len(df)*0.2), 'revenue')
print(f"  Top 20% generate ${top_20['revenue'].sum():,.2f} ({top_20['revenue'].sum()/df['revenue'].sum()*100:.1f}% of total)")

print(f"\nChurn Rate: {df['churn'].mean()*100:.1f}% ({df['churn'].sum()} out of {len(df)})")

high_cancel = df[df['cancellations_count'] >= 3]
print(f"\n3+ cancellations churn rate: {high_cancel['churn'].mean()*100:.1f}% ({len(high_cancel)} customers)")

print(f"\nTop Category: {best_category}")
print(f"Top Country: {best_country}")
print(f"\nRFM Segments: Champions={champions}, At Risk={at_risk}, Hibernating={hibernating}")
print(f"AOV: ${df['revenue'].mean():,.2f}")
print(f"Paused churn rate: {df[df['subscription_status']=='paused']['churn'].mean()*100:.1f}%")
print(f"\nModel Accuracy: {accuracy:.1%}, ROC-AUC: {roc_auc:.4f}")
print(f"High Risk revenue: ${df[df['risk_segment']=='High Risk']['revenue'].sum():,.2f}")

## Business Recommendations

In [ ]:
print("Business Recommendations:")
print()
print("1. Priority Retention: Contact top 50 high-revenue, high-churn-probability customers.")
print(f"
{df[df['subscription_status']=='paused'].shape[0]} paused customers.")
print(f"3. Investigate friction for customers with 3+ cancellations ({high_cancel['churn'].mean()*100:.1f}% churn rate).")
print(f"4. Category-specific offers (top category: {best_category}).")
print(f"5. Country-specific strategies (top country: {best_country}).")
print(f"6. Loyalty program for {champions} Champions.")
print("7. Automated monitoring system for early risk detection.")

## Executive Summary

### Business Problem
An e-commerce company selling products across 6 countries and 5 categories needed to understand sales performance, identify churn patterns, and build a predictive model for at-risk customers.

### Dataset
- 2,000 customer records, 17 columns, 6 countries, 5 categories
- No missing values or duplicate rows
- Each customer has exactly 1 order

### Major Findings
- **Total Revenue**: $2,051,690.65 across 2,000 orders
- **Churn Rate**: 24.6% (493 customers cancelled)
- **AOV**: $1,025.85
- **Top Category**: Clothing
- **Top Country**: Germany

### Customer Insights
- **RFM Segments**: 320 Champions, 293 At Risk, 31 Hibernating
- **Risk Factor**: Customers with 3+ cancellations have 24.8% churn rate
- **Model**: Logistic Regression achieved 75.2% accuracy, 0.5072 ROC-AUC

### Recommended Actions
1. Priority retention for high-value high-risk customers
2. Re-engagement campaigns for paused subscribers
3. Investigate product friction for high-cancellation customers
4. Category-specific and country-specific retention strategies
5. Loyalty program for Champions segment
6. Automated early-warning monitoring system

## Conclusion

This project demonstrates a complete data science workflow from data loading and quality audit through descriptive, diagnostic, predictive, and prescriptive analytics.

**Key Takeaways:**

1. **Data Quality**: Clean dataset with no missing values or duplicates. Each customer has exactly 1 order.

2. **Descriptive Analytics**: Revenue analysis revealed category and country patterns that inform business strategy.

3. **Diagnostic Analytics**: Cancellation count and subscription status are the strongest predictors of churn.

4. **RFM Analysis**: Customer segmentation using Recency, Frequency, and Monetary scores provided actionable segments.

5. **Predictive Modeling**: Logistic Regression identified at-risk customers with interpretable coefficients.

6. **Prescriptive Analytics**: Risk-based prioritization strategies were evaluated for the retention team.

**Limitations:**
- Churn defined via subscription status rather than temporal no-purchase window (single-order-per-customer structure)
- Cross-sectional features only; longitudinal data would strengthen findings
- All findings are correlational, not causal

**Future Work:**
- Try Random Forest, XGBoost for comparison
- Add product-level feature engineering
- Implement A/B testing for retention campaigns
- Build a monitoring dashboard